In [5]:
# PARAMETERS CELL — defaults here, overridden by the Data Pipeline later
batch_size = 3          # start small for the first test run
years_to_load = "2015,2016,2017"
reset_state = False     # True wipes progress and starts over

StatementMeta(, a0f671ec-402d-457c-8c64-23ed6348c04e, 7, Finished, Available, Finished, False)

In [6]:
import requests
import io
import zipfile
from datetime import datetime, timezone
from pyspark.sql import Row

# Month is NOT zero-padded in the URL: March 2016 = "..._2016_3.zip"
ONTIME_URL = ("https://transtats.bts.gov/PREZIP/"
              "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip")
T100_URL = "https://transtats.bts.gov/PREZIP/T_T100D_SEGMENT_ALL_CARRIER_{year}.zip"

# BTS rejects the default python-requests User-Agent, so present as a browser.
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

# Raw CSVs go in Files (not Tables) because bronze is untyped and inconsistent.
BRONZE_ONTIME = "Files/bronze/on_time"
BRONZE_T100   = "Files/bronze/t100"

# State lives in a Delta table: the pipeline is stateless between runs and
# has no local disk, so state must be durable and external.
STATE_TABLE = "pipeline_state"

YEARS  = [int(y) for y in years_to_load.split(",")]
MONTHS = list(range(1, 13))

print(f"Config: years={YEARS}, batch_size={batch_size}")

StatementMeta(, a0f671ec-402d-457c-8c64-23ed6348c04e, 8, Finished, Available, Finished, False)

Config: years=[2015, 2016, 2017], batch_size=3


In [4]:
# Delete on_time bronze only — T-100 stays (manually uploaded, hard to replace)
try:
    mssparkutils.fs.rm("Files/bronze/on_time", recurse=True)
    print("Deleted Files/bronze/on_time")
except Exception as e:
    print(f"on_time: {e}")

# Clear state, then re-mark T-100 as done so it isn't retried
spark.sql("DROP TABLE IF EXISTS pipeline_state")
print("State table dropped")

for t in ["silver_flights", "silver_tonnage",
          "gold_dim_date", "gold_dim_carrier", "gold_dim_route",
          "gold_fact_flights", "gold_fact_tonnage"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
    print(f"Dropped {t}")

print("\nNow run cells 1-3 to recreate the state table, then the snippet below.")

StatementMeta(, a0f671ec-402d-457c-8c64-23ed6348c04e, 6, Finished, Available, Finished, False)

Deleted Files/bronze/on_time
State table dropped
Dropped silver_flights
Dropped silver_tonnage
Dropped gold_dim_date
Dropped gold_dim_carrier
Dropped gold_dim_route
Dropped gold_fact_flights
Dropped gold_fact_tonnage

Now run cells 1-3 to recreate the state table, then the snippet below.


In [7]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, TimestampType
)

# Explicit schema — must match the CREATE TABLE types exactly.
# Without it, Spark infers Python int as LongType (BIGINT) and the append
# fails against an INT column. Inference guesses; declaring is deterministic.
STATE_SCHEMA = StructType([
    StructField("item_key",      StringType()),
    StructField("item_type",     StringType()),
    StructField("year",          IntegerType()),
    StructField("month",         IntegerType()),
    StructField("status",        StringType()),
    StructField("rows_ingested", LongType()),
    StructField("ingested_at",   TimestampType()),
])


def ensure_state_table():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
            item_key      STRING,
            item_type     STRING,
            year          INT,
            month         INT,
            status        STRING,
            rows_ingested BIGINT,
            ingested_at   TIMESTAMP
        ) USING DELTA
    """)


def get_completed() -> set:
    try:
        rows = spark.sql(
            f"SELECT item_key FROM {STATE_TABLE} WHERE status = 'completed'"
        ).collect()
        return {r.item_key for r in rows}
    except Exception:
        return set()


def record(item_key, item_type, year, month, status, rows=0):
    # Append-only, so failures stay in the history. That history is the audit trail.
    df = spark.createDataFrame(
        [(item_key, item_type, int(year), int(month), status,
          int(rows), datetime.now(timezone.utc))],
        schema=STATE_SCHEMA
    )
    df.write.format("delta").mode("append").saveAsTable(STATE_TABLE)


def build_plan():
    return [(y, m) for y in sorted(YEARS) for m in MONTHS]


def key(y, m):
    return f"{y}-{m:02d}"


# Drop any table left over from the failed first attempt, so we don't
# inherit its schema.
spark.sql(f"DROP TABLE IF EXISTS {STATE_TABLE}")

ensure_state_table()

if reset_state:
    spark.sql(f"DELETE FROM {STATE_TABLE}")
    print("State cleared.")

completed = get_completed()
print(f"Already completed: {len(completed)} items")

StatementMeta(, a0f671ec-402d-457c-8c64-23ed6348c04e, 9, Finished, Available, Finished, False)

Already completed: 0 items


In [8]:
for y in [2015, 2016, 2017]:
    record(f"t100-{y}", "t100", y, 0, "completed", 0)
print("T-100 re-marked as ingested.")

StatementMeta(, a0f671ec-402d-457c-8c64-23ed6348c04e, 10, Finished, Available, Finished, False)

T-100 re-marked as ingested.


In [9]:
# What's left in bronze?
for f in ["Files/bronze/on_time", "Files/bronze/t100"]:
    try:
        print(f, "->", [x.name for x in mssparkutils.fs.ls(f)])
    except Exception as e:
        print(f, "-> gone/empty")

# What does state say now?
spark.sql("SELECT item_key, item_type, status FROM pipeline_state ORDER BY item_key").show(50)

# What tables exist?
spark.sql("SHOW TABLES").show()

StatementMeta(, a0f671ec-402d-457c-8c64-23ed6348c04e, 11, Finished, Available, Finished, False)

Files/bronze/on_time -> gone/empty
Files/bronze/t100 -> ['year=2015', 'year=2016', 'year=2017']
+---------+---------+---------+
| item_key|item_type|   status|
+---------+---------+---------+
|t100-2015|     t100|completed|
|t100-2016|     t100|completed|
|t100-2017|     t100|completed|
+---------+---------+---------+

+--------------------+--------------+-----------+
|           namespace|     tableName|isTemporary|
+--------------------+--------------+-----------+
|`Cargo-Ops-Analyt...|pipeline_state|      false|
+--------------------+--------------+-----------+



In [4]:
def download_and_land(url, dest_path, label):
    """Fetch a zip from BTS, extract the CSV, write it to OneLake."""
    try:
        # verify=False: BTS has certificate issues — every public example of
        # scripting their downloads disables verification.
        resp = requests.get(url, headers=HEADERS, timeout=300, verify=False)

        if resp.status_code == 404:
            print(f"  {label}: NOT FOUND (404)")
            return False, 0
        resp.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
            if not csv_names:
                print(f"  {label}: no CSV inside the zip")
                return False, 0
            # If multiple, take the largest — the data file, not a readme.
            csv_names.sort(key=lambda n: zf.getinfo(n).file_size, reverse=True)
            data = zf.read(csv_names[0])

        mssparkutils.fs.put(dest_path, data.decode("latin-1"), overwrite=True)

        rows = data.count(b"\n") - 1
        print(f"  {label}: OK — {rows:,} rows, {len(data)/1_048_576:.1f} MB")
        return True, rows

    except Exception as e:
        print(f"  {label}: FAILED — {type(e).__name__}: {e}")
        return False, 0

StatementMeta(, f73340c8-3622-4c15-97cd-1d7d938cd1de, 6, Finished, Available, Finished, False)

In [5]:
plan = build_plan()
pending = [(y, m) for y, m in plan if key(y, m) not in completed]

print("=" * 60)
print(f"BATCH RUN — {len(pending)} months pending, taking {batch_size}")
print("=" * 60)

# A finished pipeline must SUCCEED, not error, or the scheduler reports
# failures forever once the backfill is done.
if not pending:
    print("Nothing pending — backfill complete. Clean no-op.")
else:
    batch = pending[:batch_size]
    print(f"Ingesting {key(*batch[0])} through {key(*batch[-1])}\n")

    for y, m in batch:
        url  = ONTIME_URL.format(year=y, month=m)                     # unpadded
        dest = f"{BRONZE_ONTIME}/year={y}/month={m:02d}/on_time.csv"   # padded
        ok, rows = download_and_land(url, dest, f"on_time {y}-{m:02d}")
        record(key(y, m), "ontime", y, m, "completed" if ok else "failed", rows)

    # T-100 is ANNUAL — fetch a year's file once all 12 months are in.
    completed = get_completed()
    for y in sorted(YEARS):
        t_key = f"t100-{y}"
        if t_key in completed:
            continue
        if all(key(y, m) in completed for m in MONTHS):
            print(f"\n{y} complete — fetching its T-100 file")
            ok, rows = download_and_land(
                T100_URL.format(year=y),
                f"{BRONZE_T100}/year={y}/t100_segment.csv",
                f"t100 {y}")
            record(t_key, "t100", y, 0, "completed" if ok else "failed", rows)

StatementMeta(, f73340c8-3622-4c15-97cd-1d7d938cd1de, 7, Finished, Available, Finished, False)

BATCH RUN — 36 months pending, taking 12
Ingesting 2015-01 through 2015-12

  on_time 2015-01: OK — 469,968 rows, 201.8 MB
  on_time 2015-02: OK — 429,191 rows, 183.9 MB
  on_time 2015-04: OK — 485,151 rows, 208.5 MB
  on_time 2015-05: OK — 496,993 rows, 213.6 MB
  on_time 2015-06: OK — 503,897 rows, 216.9 MB
  on_time 2015-08: OK — 510,536 rows, 219.5 MB
  on_time 2015-09: OK — 464,946 rows, 199.5 MB
  on_time 2015-10: OK — 486,165 rows, 209.0 MB
  on_time 2015-12: OK — 479,230 rows, 206.5 MB
  t100 2015: NOT FOUND (404)


/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'transtats.bts.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'transtats.bts.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'transtats.bts.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/e

In [6]:
completed = get_completed()
months_done = len([c for c in completed if not c.startswith("t100")])
t100_done   = len([c for c in completed if c.startswith("t100")])
total = len(build_plan())

pct = months_done / total * 100
filled = int(30 * months_done / total)
bar = "#" * filled + "-" * (30 - filled)

print("=" * 60)
print(f"  [{bar}]  {pct:.0f}%")
print(f"  Months: {months_done}/{total}    T-100 files: {t100_done}/{len(YEARS)}")
print("=" * 60)

spark.sql(f"""
    SELECT item_key, item_type, status, rows_ingested, ingested_at
    FROM {STATE_TABLE}
    ORDER BY ingested_at DESC
    LIMIT 20
""").show(truncate=False)

StatementMeta(, f73340c8-3622-4c15-97cd-1d7d938cd1de, 8, Finished, Available, Finished, False)

  [##########--------------------]  33%
  Months: 12/36    T-100 files: 0/3
+---------+---------+---------+-------------+--------------------------+
|item_key |item_type|status   |rows_ingested|ingested_at               |
+---------+---------+---------+-------------+--------------------------+
|t100-2015|t100     |failed   |0            |2026-08-30 11:43:18.543136|
|2015-12  |ontime   |completed|479230       |2026-08-30 11:43:07.361208|
|2015-11  |ontime   |completed|467972       |2026-08-30 11:42:07.86207 |
|2015-10  |ontime   |completed|486165       |2026-08-30 11:41:05.49315 |
|2015-09  |ontime   |completed|464946       |2026-08-30 11:40:13.776084|
|2015-08  |ontime   |completed|510536       |2026-08-30 11:39:10.590776|
|2015-07  |ontime   |completed|520718       |2026-08-30 11:38:17.900245|
|2015-06  |ontime   |completed|503897       |2026-08-30 11:37:24.435416|
|2015-05  |ontime   |completed|496993       |2026-08-30 11:36:26.560785|
|2015-04  |ontime   |completed|485151       |202

In [7]:
import requests
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

candidates = [
    "https://transtats.bts.gov/PREZIP/T_T100D_SEGMENT_ALL_CARRIER_2015.zip",
    "https://transtats.bts.gov/PREZIP/T_T100D_SEGMENT_ALL_CARRIER.zip",
    "https://transtats.bts.gov/PREZIP/T_T100_SEGMENT_ALL_CARRIER_2015.zip",
    "https://transtats.bts.gov/PREZIP/T_T100D_SEGMENT_US_CARRIER_ONLY_2015.zip",
]

for url in candidates:
    try:
        r = requests.head(url, headers=HEADERS, timeout=60, verify=False, allow_redirects=True)
        print(f"{r.status_code}  {url}")
    except Exception as e:
        print(f"ERR   {url}  ({type(e).__name__})")

StatementMeta(, f73340c8-3622-4c15-97cd-1d7d938cd1de, 9, Finished, Available, Finished, False)

404  https://transtats.bts.gov/PREZIP/T_T100D_SEGMENT_ALL_CARRIER_2015.zip
404  https://transtats.bts.gov/PREZIP/T_T100D_SEGMENT_ALL_CARRIER.zip
404  https://transtats.bts.gov/PREZIP/T_T100_SEGMENT_ALL_CARRIER_2015.zip
404  https://transtats.bts.gov/PREZIP/T_T100D_SEGMENT_US_CARRIER_ONLY_2015.zip


In [8]:
for y in [2015, 2016, 2017]:
    mssparkutils.fs.mkdirs(f"Files/bronze/t100/year={y}")
print("Folders created.")
mssparkutils.fs.ls("Files/bronze/t100")

StatementMeta(, f73340c8-3622-4c15-97cd-1d7d938cd1de, 10, Finished, Available, Finished, False)

Folders created.


[FileInfo(path=abfss://8a0ad676-453e-4aae-bf78-9c1fdca19d2b@onelake.dfs.fabric.microsoft.com/00e45c05-bf3a-40cd-9c2f-9f6428b6182b/Files/bronze/t100/year=2015, name=year=2015, size=0),
 FileInfo(path=abfss://8a0ad676-453e-4aae-bf78-9c1fdca19d2b@onelake.dfs.fabric.microsoft.com/00e45c05-bf3a-40cd-9c2f-9f6428b6182b/Files/bronze/t100/year=2016, name=year=2016, size=0),
 FileInfo(path=abfss://8a0ad676-453e-4aae-bf78-9c1fdca19d2b@onelake.dfs.fabric.microsoft.com/00e45c05-bf3a-40cd-9c2f-9f6428b6182b/Files/bronze/t100/year=2017, name=year=2017, size=0)]